# Placebo-Anchored Transfer Learning: Benchmark Sweeps

This notebook runs the core benchmark sweeps for evaluating the placebo-anchored DR-Learner with sparse corrections.

**Sweeps included:**
1. **Gold sweep (m₀)**: Target placebo budget
2. **Proxy sweep (n_proxy)**: Source data budget
3. **Site imbalance**: Robustness to unequal site sizes

**Methods compared:**
- `NoTransfer`: Target-only DR-Learner (no transfer)
- `ProxyOnly`: Source-only estimator (ignores target)
- `AnchorOnly`: Target placebo correction only
- `ProposedA`: Full method with target treated data
- `ProposedB_LinearStepB`: Placebo-anchored with Step B transfer

## 1. Setup

In [ ]:
# Configuration
N_REPS = 100           # Number of Monte Carlo replications (increase for production)
N_JOBS = -1            # Number of parallel jobs (-1 = all cores)
OUTPUT_DIR = "../results/sweeps_notebook"
SEED = 42

# Which sweeps to run
RUN_GOLD = True
RUN_PROXY = True
RUN_IMBALANCE = True

In [ ]:
import os
import sys
import warnings
import time
from datetime import datetime

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('../experiments'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress convergence warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*ConvergenceWarning.*')

# Set matplotlib backend for inline display
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# Import sweep functions
from core_sweeps import (
    run_sweep, run_all_sweeps,
    generate_sweep_plots, generate_sweep_report,
    SWEEP_CONFIGS, DEFAULT_METHODS
)
from benchmark_plots import setup_plot_style, plot_line, METHOD_COLORS, METHOD_MARKERS
from benchmark_aggregation import aggregate_results, find_best_methods, compute_method_rankings

print("Imports successful!")
print(f"\nAvailable sweeps: {list(SWEEP_CONFIGS.keys())}")
print(f"Default methods: {DEFAULT_METHODS}")

## 2. Run Sweeps

Each sweep runs in parallel using all available CPU cores. Progress is displayed via tqdm.

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Store results
results = {}
start_time = time.time()

### 2.1 Gold Sweep (Target Placebo Budget)

In [ ]:
if RUN_GOLD:
    print("="*70)
    print("Running Gold Sweep (m₀ budget)")
    print("="*70)
    
    t0 = time.time()
    df_rep_gold, df_agg_gold = run_sweep(
        'gold',
        n_rep=N_REPS,
        seed0=SEED,
        output_dir=OUTPUT_DIR,
        n_jobs=N_JOBS,
        verbose=True
    )
    results['gold'] = (df_rep_gold, df_agg_gold)
    print(f"\nGold sweep completed in {time.time()-t0:.1f}s")
else:
    print("Gold sweep skipped (RUN_GOLD=False)")

### 2.2 Proxy Sweep (Source Data Budget)

In [ ]:
if RUN_PROXY:
    print("="*70)
    print("Running Proxy Sweep (n_proxy budget)")
    print("="*70)
    
    t0 = time.time()
    df_rep_proxy, df_agg_proxy = run_sweep(
        'proxy',
        n_rep=N_REPS,
        seed0=SEED,
        output_dir=OUTPUT_DIR,
        n_jobs=N_JOBS,
        verbose=True
    )
    results['proxy'] = (df_rep_proxy, df_agg_proxy)
    print(f"\nProxy sweep completed in {time.time()-t0:.1f}s")
else:
    print("Proxy sweep skipped (RUN_PROXY=False)")

### 2.3 Site Imbalance Sweep

In [ ]:
if RUN_IMBALANCE:
    print("="*70)
    print("Running Site Imbalance Sweep")
    print("="*70)
    
    t0 = time.time()
    df_rep_imb, df_agg_imb = run_sweep(
        'imbalance',
        n_rep=N_REPS,
        seed0=SEED,
        output_dir=OUTPUT_DIR,
        n_jobs=N_JOBS,
        verbose=True
    )
    results['imbalance'] = (df_rep_imb, df_agg_imb)
    print(f"\nImbalance sweep completed in {time.time()-t0:.1f}s")
else:
    print("Imbalance sweep skipped (RUN_IMBALANCE=False)")

In [ ]:
total_time = time.time() - start_time
print(f"\n{'='*70}")
print(f"All sweeps completed!")
print(f"Total runtime: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"Results saved to: {OUTPUT_DIR}")
print(f"{'='*70}")

## 3. Results Summary

In [ ]:
def display_best_methods(df_agg, sweep_name):
    """Display best methods for a sweep."""
    best = find_best_methods(df_agg, metrics=['pehe', 'ate_abs_err', 'tau_corr'],
                            lower_is_better={'pehe': True, 'ate_abs_err': True, 'tau_corr': False})
    
    print(f"\n📊 Best Methods for {sweep_name.upper()} Sweep:")
    print("-" * 50)
    for metric, info in best.items():
        direction = "↓" if metric in ['pehe', 'ate_abs_err'] else "↑"
        print(f"  {metric:15s} {direction}: {info['method']:25s} ({info['value']:.4f})")
    return best

In [ ]:
# Display summary for each sweep
for sweep_name, (df_rep, df_agg) in results.items():
    display_best_methods(df_agg, sweep_name)

## 4. Visualizations

### 4.1 PEHE Comparison

In [ ]:
def plot_sweep_results(df_agg, sweep_param, title, figsize=(12, 4)):
    """Plot PEHE, ATE error, and correlation for a sweep."""
    setup_plot_style()
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # PEHE
    ax = axes[0]
    for method in df_agg['method'].unique():
        subset = df_agg[df_agg['method'] == method].sort_values(sweep_param)
        color = METHOD_COLORS.get(method, '#333333')
        marker = METHOD_MARKERS.get(method, 'o')
        ax.errorbar(subset[sweep_param], subset['pehe_mean'], 
                   yerr=subset['pehe_sd'], label=method,
                   color=color, marker=marker, capsize=3, markersize=6)
    ax.set_xlabel(sweep_param)
    ax.set_ylabel('PEHE')
    ax.set_title('PEHE (↓ lower is better)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # ATE Error
    ax = axes[1]
    for method in df_agg['method'].unique():
        subset = df_agg[df_agg['method'] == method].sort_values(sweep_param)
        color = METHOD_COLORS.get(method, '#333333')
        marker = METHOD_MARKERS.get(method, 'o')
        if 'ate_abs_err_mean' in subset.columns:
            ax.errorbar(subset[sweep_param], subset['ate_abs_err_mean'], 
                       yerr=subset['ate_abs_err_sd'], label=method,
                       color=color, marker=marker, capsize=3, markersize=6)
    ax.set_xlabel(sweep_param)
    ax.set_ylabel('|ATE Error|')
    ax.set_title('ATE Error (↓ lower is better)')
    ax.grid(True, alpha=0.3)
    
    # Correlation
    ax = axes[2]
    for method in df_agg['method'].unique():
        subset = df_agg[df_agg['method'] == method].sort_values(sweep_param)
        color = METHOD_COLORS.get(method, '#333333')
        marker = METHOD_MARKERS.get(method, 'o')
        if 'tau_corr_mean' in subset.columns:
            ax.errorbar(subset[sweep_param], subset['tau_corr_mean'], 
                       yerr=subset['tau_corr_sd'], label=method,
                       color=color, marker=marker, capsize=3, markersize=6)
    ax.set_xlabel(sweep_param)
    ax.set_ylabel('Spearman ρ')
    ax.set_title('Rank Correlation (↑ higher is better)')
    ax.grid(True, alpha=0.3)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

In [ ]:
# Gold sweep visualization
if 'gold' in results:
    df_rep, df_agg = results['gold']
    fig = plot_sweep_results(df_agg, 'm0', 'Gold Sweep: Target Placebo Budget (m₀)', figsize=(14, 4))
    plt.show()

In [ ]:
# Proxy sweep visualization
if 'proxy' in results:
    df_rep, df_agg = results['proxy']
    fig = plot_sweep_results(df_agg, 'n_proxy_total', 'Proxy Sweep: Source Data Budget', figsize=(14, 4))
    plt.show()

In [ ]:
# Imbalance sweep visualization
if 'imbalance' in results:
    df_rep, df_agg = results['imbalance']
    fig = plot_sweep_results(df_agg, 'imbalance_ratio', 'Site Imbalance Sweep', figsize=(14, 4))
    plt.show()

### 4.2 Method Rankings

In [ ]:
def plot_rankings_heatmap(results_dict):
    """Plot method rankings across all sweeps."""
    all_rankings = []
    
    for sweep_name, (df_rep, df_agg) in results_dict.items():
        rankings = compute_method_rankings(
            df_agg, 
            metrics=['pehe', 'ate_abs_err', 'tau_corr'],
            lower_is_better={'pehe': True, 'ate_abs_err': True, 'tau_corr': False}
        )
        if not rankings.empty:
            rankings['sweep'] = sweep_name
            all_rankings.append(rankings.reset_index())
    
    if not all_rankings:
        print("No rankings to display")
        return
    
    combined = pd.concat(all_rankings)
    
    # Average across sweeps
    avg_rankings = combined.groupby('method')[['pehe', 'ate_abs_err', 'tau_corr']].mean()
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(avg_rankings, annot=True, fmt='.2f', cmap='RdYlGn_r', 
                ax=ax, cbar_kws={'label': 'Average Rank (lower is better)'})
    ax.set_title('Method Rankings Across All Sweeps', fontweight='bold')
    ax.set_xlabel('Metric')
    ax.set_ylabel('Method')
    plt.tight_layout()
    return fig

if results:
    fig = plot_rankings_heatmap(results)
    plt.show()

### 4.3 Distribution of PEHE (Violin Plots)

In [ ]:
def plot_pehe_distributions(results_dict):
    """Plot PEHE distributions for each sweep."""
    n_sweeps = len(results_dict)
    if n_sweeps == 0:
        return
    
    fig, axes = plt.subplots(1, n_sweeps, figsize=(5*n_sweeps, 5))
    if n_sweeps == 1:
        axes = [axes]
    
    for ax, (sweep_name, (df_rep, df_agg)) in zip(axes, results_dict.items()):
        # Use rep-level data for distributions
        palette = {m: METHOD_COLORS.get(m, '#333333') for m in df_rep['method'].unique()}
        sns.violinplot(data=df_rep, x='method', y='pehe', palette=palette, ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_title(f'{sweep_name.capitalize()} Sweep')
        ax.set_ylabel('PEHE')
        ax.set_xlabel('')
    
    fig.suptitle('PEHE Distributions by Method', fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

if results:
    fig = plot_pehe_distributions(results)
    plt.show()

## 5. Detailed Results Tables

In [ ]:
def format_results_table(df_agg, sweep_param):
    """Format aggregated results for display."""
    cols = [sweep_param, 'method', 'pehe_mean', 'pehe_sd', 'ate_abs_err_mean', 'tau_corr_mean']
    cols = [c for c in cols if c in df_agg.columns]
    
    display_df = df_agg[cols].copy()
    
    # Round numeric columns
    for col in display_df.columns:
        if display_df[col].dtype in ['float64', 'float32']:
            display_df[col] = display_df[col].round(4)
    
    return display_df.sort_values([sweep_param, 'method'])

In [ ]:
# Gold sweep table
if 'gold' in results:
    print("\n" + "="*70)
    print("GOLD SWEEP RESULTS (m₀ budget)")
    print("="*70)
    display(format_results_table(results['gold'][1], 'm0'))

In [ ]:
# Proxy sweep table
if 'proxy' in results:
    print("\n" + "="*70)
    print("PROXY SWEEP RESULTS (n_proxy budget)")
    print("="*70)
    display(format_results_table(results['proxy'][1], 'n_proxy_total'))

In [ ]:
# Imbalance sweep table
if 'imbalance' in results:
    print("\n" + "="*70)
    print("SITE IMBALANCE SWEEP RESULTS")
    print("="*70)
    display(format_results_table(results['imbalance'][1], 'imbalance_ratio'))

## 6. Export Results

In [ ]:
# Generate plots and reports for each sweep
for sweep_name, (df_rep, df_agg) in results.items():
    print(f"\nGenerating outputs for {sweep_name} sweep...")
    generate_sweep_plots(sweep_name, df_agg, OUTPUT_DIR, verbose=True)
    report_path = generate_sweep_report(sweep_name, df_rep, df_agg, OUTPUT_DIR)
    print(f"  Report: {report_path}")

In [ ]:
# List output files
print(f"\n📁 Output files in {OUTPUT_DIR}:")
print("-" * 50)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:40s} ({size/1024:.1f} KB)")

## 7. Key Findings Summary

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

for sweep_name, (df_rep, df_agg) in results.items():
    print(f"\n### {sweep_name.upper()} SWEEP ###")
    
    # Best PEHE method
    best_pehe_idx = df_agg['pehe_mean'].idxmin()
    if not pd.isna(best_pehe_idx):
        best_row = df_agg.loc[best_pehe_idx]
        print(f"  Best PEHE: {best_row['method']} ({best_row['pehe_mean']:.4f})")
    
    # ProposedB vs ProxyOnly improvement
    proposed_pehe = df_agg[df_agg['method'] == 'ProposedB_LinearStepB']['pehe_mean'].mean()
    proxy_pehe = df_agg[df_agg['method'] == 'ProxyOnly']['pehe_mean'].mean()
    if not np.isnan(proposed_pehe) and not np.isnan(proxy_pehe):
        improvement = (proxy_pehe - proposed_pehe) / proxy_pehe * 100
        print(f"  ProposedB vs ProxyOnly: {improvement:+.1f}% PEHE {'improvement' if improvement > 0 else 'change'}")
    
    # ProposedB vs AnchorOnly improvement
    anchor_pehe = df_agg[df_agg['method'] == 'AnchorOnly']['pehe_mean'].mean()
    if not np.isnan(proposed_pehe) and not np.isnan(anchor_pehe):
        improvement = (anchor_pehe - proposed_pehe) / anchor_pehe * 100
        print(f"  ProposedB vs AnchorOnly: {improvement:+.1f}% PEHE {'improvement' if improvement > 0 else 'change'}")

print("\n" + "="*70)
print(f"Notebook completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total runtime: {total_time/60:.1f} minutes")
print("="*70)